# 01 — Ingest Sources

Validate the five manifest-declared landing artifacts and emit an auditable collection result for each source. Inputs are read from the Unity Catalog `raw_sources` volume; outputs are validation events and source inspections consumed by Bronze.

Databricks Free Edition cannot reach the Madrid source domain, so this execution profile validates two local files and three complete remote-origin snapshots uploaded by the legacy workflow. Direct HTTP collection remains a future environment capability.

## 1. Runtime and source catalog

Reusable collection logic lives in `src/urban_workflow/ingestion`; `config/source_catalog.json` declares source identity, access mode, format, and lineage. The notebook supplies development catalog, schema, and volume values, validates the manifest, and confirms access to the existing Unity Catalog namespace.

The combined GeoPackage remains explicitly classified as a `derived_snapshot`. The bundle-managed Lakeflow Job supplies the same runtime values through notebook parameters.

In [ ]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in [cwd, *cwd.parents] if (candidate / "src" / "urban_workflow").exists()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Could not locate the bundle root containing src/urban_workflow.")

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from urban_workflow.ingestion.collect_local_source import collect_local_source
from urban_workflow.ingestion.collect_remote_source import validate_manually_uploaded_remote_source
from urban_workflow.ingestion.schemas import (
    CollectionEvent,
    SourceManifest,
    resolve_manifest_paths,
)

print(f"Bundle root: {repo_root}")

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "urban_workflow")
dbutils.widgets.text("raw_volume", "raw_sources")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()
RAW_VOLUME = dbutils.widgets.get("raw_volume").strip()
if not all((CATALOG, SCHEMA, RAW_VOLUME)):
    raise ValueError("catalog, schema, and raw_volume must be non-empty.")

VOLUME_FQN = f"{CATALOG}.{SCHEMA}.{RAW_VOLUME}"
RAW_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{RAW_VOLUME}"

print(f"Volume: {VOLUME_FQN}")
print(f"Landing root: {RAW_ROOT}")

In [ ]:
manifest_path = repo_root / "config" / "source_catalog.json"
with manifest_path.open("r", encoding="utf-8") as file_handle:
    manifest = SourceManifest.model_validate(json.load(file_handle))
manifest = resolve_manifest_paths(manifest, RAW_ROOT)

source_overview = [
    {
        "source_id": source.source_id,
        "acquisition_mode": source.acquisition_mode,
        "source_kind": source.source_kind,
        "source_type": source.source_type,
        "file_format": source.file_format,
        "enabled": source.enabled,
    }
    for source in manifest.sources
]

print(f"Validated {len(source_overview)} source specifications from {manifest_path.name}.")
display(spark.createDataFrame(source_overview))

In [ ]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
spark.sql("SELECT current_catalog() AS catalog, current_schema() AS schema").show(truncate=False)
spark.sql(f"DESCRIBE VOLUME {VOLUME_FQN}").show(truncate=False)

In [ ]:
display(dbutils.fs.ls(RAW_ROOT))

## 2. Collection evidence

One collection event records the outcome for one source in one run, including status, method, record count, warnings, landed path, and provenance. This contract makes successful validation and failure evidence queryable instead of leaving it in notebook logs.

SHA-256 fingerprints every byte: matching hashes strongly indicate identical content, while a changed hash exposes source drift even if the filename is unchanged.

In [ ]:
collection_event_fields = [
    {
        "field": field_name,
        "required": field.is_required(),
        "annotation": str(field.annotation),
    }
    for field_name, field in CollectionEvent.model_fields.items()
]
display(spark.createDataFrame(collection_event_fields))

## 3. Validate local sources

The local dispatcher validates the combined-grid GeoPackage and EMVS CSV according to their manifest formats. GeoPackage inspection reads its SQLite catalog in read-only mode to obtain layers, schema, geometry type, CRS, and row count without loading all polygons.

Before format inspection, the collector rejects unresolved Git LFS pointers: small text references to missing large-object bytes. A path can therefore exist while the actual dataset is absent.

In [ ]:
grid_source = next(
    source for source in manifest.sources
    if source.source_id == "madrid_combined_grid_snapshot"
)
run_id = (
    f"interactive_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_"
    f"{uuid4().hex[:8]}"
)

grid_result = collect_local_source(
    specification=grid_source,
    run_id=run_id,
    landing_root=Path(RAW_ROOT),
)

print(grid_result.event.model_dump_json(indent=2))
if grid_result.event.status != "succeeded":
    raise RuntimeError(grid_result.event.error_message)

In [ ]:
inspection = grid_result.geopackage
assert inspection is not None

print(inspection.model_dump_json(indent=2))
column_overview = [
    {
        "column": column,
        "declared_type": inspection.column_types[column],
        "is_geometry": column == inspection.geometry_column,
    }
    for column in inspection.columns
]
display(spark.createDataFrame(column_overview))

### EMVS CSV

The manifest declares UTF-8 with BOM, a semicolon delimiter, and a header after one preamble row. Validation retains all 22 records, including `TOTAL`, and reports the two unnamed trailing columns; removing them is a Silver decision.

In [ ]:
emvs_source = next(
    source for source in manifest.sources
    if source.source_id == "madrid_emvs_housing"
)
emvs_result = collect_local_source(
    specification=emvs_source,
    run_id=run_id,
    landing_root=Path(RAW_ROOT),
)

print(emvs_result.event.model_dump_json(indent=2))
if emvs_result.event.status != "succeeded":
    raise RuntimeError(emvs_result.event.error_message)

In [ ]:
csv_inspection = emvs_result.csv
assert csv_inspection is not None

print(csv_inspection.model_dump_json(indent=2))
csv_column_overview = [
    {
        "position": position,
        "source_column": column or "<empty>",
        "landing_type": "string",
        "empty_name": not column,
    }
    for position, column in enumerate(csv_inspection.columns, start=1)
]
display(spark.createDataFrame(csv_column_overview))

## 4. Validate remote-origin snapshots

The two CKAN responses and district-boundary TopoJSON retain their API or remote-file origins while `collection_method=manual_upload` records how these snapshots entered this Free Edition workspace. This distinction avoids claiming that Databricks performed the blocked HTTP requests.

CKAN snapshots are accepted only when the stored records equal `result.total`, preventing a first-page sample from entering Bronze. The boundary must be valid TopoJSON with nonempty objects and an arcs array. Every file also receives size and SHA-256 provenance.

In [ ]:
remote_source_ids = [
    "madrid_population_district_barrio_api",
    "madrid_district_indicator_panel_api",
    "madrid_district_boundaries_remote",
]
remote_results = []
for source_id in remote_source_ids:
    source_specification = next(
        source for source in manifest.sources if source.source_id == source_id
    )
    result = validate_manually_uploaded_remote_source(
        specification=source_specification,
        run_id=run_id,
        landing_root=Path(RAW_ROOT),
    )
    print(result.event.model_dump_json(indent=2))
    if result.event.status != "succeeded":
        raise RuntimeError(result.event.error_message)
    remote_results.append(result)

print(f"Validated {len(remote_results)} manually uploaded remote snapshots.")

In [ ]:
remote_validation_overview = [
    {
        "source_id": result.event.source_id,
        "acquisition_mode": result.event.acquisition_mode,
        "collection_method": result.event.collection_method,
        "record_count": result.event.record_count,
        "file_size_bytes": result.event.file_size_bytes,
        "sha256": result.event.sha256,
        "landed_path": result.event.landed_path,
    }
    for result in remote_results
]
display(spark.createDataFrame(remote_validation_overview))

for result in remote_results:
    if result.ckan_page is not None:
        print(f"{result.event.source_id} CKAN inspection:")
        print(result.ckan_page.model_dump_json(indent=2))
    if result.json_file is not None and result.json_file.root_type == "Topology":
        print(f"{result.event.source_id} TopoJSON inspection:")
        print(result.json_file.model_dump_json(indent=2))

dbutils.notebook.exit(
    f"SUCCESS: Validated {2 + len(remote_results)} sources: 2 local files and "
    f"{len(remote_results)} uploaded remote snapshots."
)

## 5. Milestone status

Milestone 1 is complete: all five landing artifacts are present, structurally validated, counted, and fingerprinted for the Free Edition execution profile. Their accepted outputs are consumed by notebook 02 to build source-shaped Bronze Delta tables.